# First backtest in a notebook

Runs the EMA-cross example on synthetic minute bars using the engine packages directly.
Requires the .NET Interactive kernel (Polyglot Notebooks). Run from the repository root so the project references resolve;
for published packages replace the `#r` project references with `#r "nuget: Bytex.Backtest"` and `#r "nuget: Bytex.Indicators"`.

In [ ]:
#r "../../src/Bytex.Core/bin/Debug/net10.0/Bytex.Core.dll"
#r "../../src/Bytex.Indicators/bin/Debug/net10.0/Bytex.Indicators.dll"
#r "../../src/Bytex.Data/bin/Debug/net10.0/Bytex.Data.dll"
#r "../../src/Bytex.Backtest/bin/Debug/net10.0/Bytex.Backtest.dll"
#r "../Bytex.Examples/bin/Debug/net10.0/Bytex.Examples.dll"

using Bytex.Backtest;
using Bytex.Core.Model;
using Bytex.Core.Model.Data;
using Bytex.Core.Model.Identifiers;
using Bytex.Core.Model.Instruments;
using Bytex.Core.Model.Primitives;
using Bytex.Examples;
using Bytex.Examples.Strategies;

In [ ]:
Instrument instrument = TestData.BtcUsdt();
BarType barType = new(instrument.Id, new BarSpecification(1, BarAggregation.Minute, PriceType.Last));
var bars = TestData.RandomWalkBars(instrument, barType, 5_000);
bars.Count

In [ ]:
using BacktestEngine engine = new();
engine.AddInstrument(instrument);
engine.AddVenue(new SimulatedVenueConfig
{
    Venue = TestData.Sim,
    AccountType = AccountType.Cash,
    StartingBalances = [new Money(1_000_000m, Currencies.USDT), new Money(10m, Currencies.BTC)],
});
engine.AddData(bars.Cast<IData>());
engine.AddStrategy(new EmaCross(new EmaCrossConfig
{
    StrategyId = new StrategyId("EmaCross-001"),
    InstrumentId = instrument.Id,
    BarType = barType,
    FastPeriod = 10,
    SlowPeriod = 30,
    TradeSize = 0.5m,
}));
engine.Run();
BacktestResult result = engine.GetResult();
Console.WriteLine(result.Summary());

In [ ]:
// Tables render as interactive grids in the notebook.
result.Positions.Take(10)

In [ ]:
// Equity curve as (timestamp, equity) pairs; plot with your preferred charting extension.
result.EquityCurves[Currencies.USDT].Select(p => new { Time = p.Timestamp.ToDateTimeUtc(), p.Equity }).TakeLast(20)